# Masterclass: the honest swap-in path

The [quickstart](quickstart.ipynb) took the short road — one score, one cutoff, the three
numbers. This is the long one. It runs on a single base, the **incumbent** book from
`generate_sample_data`, and it stays on the one problem the quickstart pointed at and walked
away from: **the outcome you cannot see.**

A book with a legacy policy in force has already rejected people, and you never found out how
those rejects would have repaid — `actual_default` is masked (`NaN`) for everyone who did not
contract. The moment a challenger policy wants to *approve* someone the incumbent rejected — a
**swap-in** — it has to put a number on a loan that was never made. Do that naively and the
number lies in the dangerous direction: it understates risk. The library says so, out loud,
mid-run.

The masterclass is where you learn to hear that warning and answer it. The arc:

1. The incumbent book and the three numbers (ADR 0008).
2. Choosing a challenger — which score actually separates risk.
3. The swap, and the calibration warning firing *for real*.
4. Measuring the damage against the oracle.
5. The two remediations the warning names — and the one that is not a fix.
6. Setting the cutoff honestly (declared direction, Pareto, per region).
7. Sequential cutoffs as a cost decision.
8. Rating, validated out of time.
9. A rate stage that is *not* take-up.
10. Suggesting the hard filters.
11. The P&L, and export.

Everything below is English — narrative, code, and every column you print.

In [1]:
import warnings

import numpy as np
import pandas as pd

from pycreditools import (
    LEGACY_APPROVAL_QUANTILE,
    CalibrationReliabilityWarning,
    CreditPolicy,
    ModelEvaluator,
    col,
    fit_risk_groups,
    generate_sample_data,
    optimize_cutoffs,
    print_delta_table,
    suggest_hard_filters,
    summarize_results,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
np.random.seed(7)

## 1. The incumbent book

`generate_sample_data` returns a book a legacy policy already runs on. Two things matter and both
are real:

- **`approved`** is the incumbent's decision — its hard knock-outs plus a cutoff on `legacy_score`.
- **`actual_default`** is *masked*: `NaN` for every applicant who did not contract. You observe
  repayment only on the loans you actually made — and that mask is the whole problem this notebook
  is about.

`true_pd` is in the frame too, but it is an **oracle**: it exists so this notebook can *measure*
calibration error instead of asserting it. No lender has it, and it is never an input to a policy.

In [2]:
base = generate_sample_data(n_applicants=60_000, seed=7)
legacy_cut = float(base["legacy_score"].quantile(LEGACY_APPROVAL_QUANTILE))

print(f"rows:                 {len(base):,}")
print(f"incumbent approval:   {base['approved'].mean():.1%}")
print(f"actual_default masked:{base['actual_default'].isna().mean():>6.1%}")
print(f"legacy cutoff:        {legacy_cut:.0f}  (legacy_score quantile {LEGACY_APPROVAL_QUANTILE})")
base[["applicant_id", "region", "legacy_score", "score_5", "hired", "actual_default"]].head()

rows:                 60,000
incumbent approval:   20.5%
actual_default masked: 90.5%
legacy cutoff:        790  (legacy_score quantile 0.78)


,applicant_id,region,legacy_score,score_5,hired,actual_default
0,1,Nordeste,104,198,0,NaN
1,2,Norte,124,160,0,NaN
2,3,Sudeste,327,193,0,NaN
3,4,Norte,306,380,0,NaN
4,5,Sudeste,483,512,0,NaN


### The three numbers, restated (ADR 0008)

Every surface in the library reports the same three, off the same two funnel columns
(`approved_pre_rate`, `new_approval`). We define the reader once and reuse it everywhere.

- **`approval_rate` = approved ÷ total** — always *pre* take-up. The underwriting rule alone.
- **`take_up_rate` = contracted ÷ approved** — a conversion, denominated in *approved*.
- **`default_rate`** — always *contracted*-weighted. You cannot default on a loan you never took.

In [3]:
def kpis(sim):
    # The ADR 0008 metric contract, read straight off the funnel columns.
    d = sim.data
    n = len(d)
    approved = d["approved_pre_rate"].sum()
    contracted = d["new_approval"].sum()
    weighted_default = (d["simulated_default"] * d["new_approval"]).sum()
    return {
        "approval_rate": approved / n,
        "take_up_rate": contracted / approved,
        "default_rate": weighted_default / contracted,
    }


incumbent = (
    CreditPolicy(
        applicant_id_col="applicant_id",
        score_cols=("score_5",),
        current_approval_col="approved",
        actual_default_col="actual_default",
    )
    .filter("Bureau knock-outs", (col("age") >= 18) & (col("vl_negativacao") <= 5000))
    .cutoff("Legacy cutoff", {"legacy_score": legacy_cut}, direction="gte")
    .rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")
)

sim_incumbent = incumbent.simulate(base, method="analytical")
for name, value in kpis(sim_incumbent).items():
    print(f"{name:>14}: {value:.3f}")

 approval_rate: 0.205
  take_up_rate: 0.464
  default_rate: 0.075


`current_approval_col="approved"` is what turns this from a standalone run into a **swap** run:
the simulator now classifies every applicant by *both* policies at once — `keep_in`, `keep_out`,
`swap_in`, `swap_out`. Re-running the incumbent against itself, there are no swaps yet.

In [4]:
sim_incumbent.data["scenario"].value_counts()

scenario
keep_out    47672
keep_in     12328
Name: count, dtype: int64

## 2. Choosing a challenger

Before swapping anyone, decide *on what*. The incumbent decides on `legacy_score`; the challenger
scores (`score_2` … `score_5`) are the models competing to replace it. The first question is not
P&L — it is separation: which score actually tells a good risk from a bad one. `ModelEvaluator`
reports KS on the population where the outcome is observed (the contracted book).

In [5]:
evaluator = ModelEvaluator(
    base, ["legacy_score", "score_2", "score_3", "score_4", "score_5"], "actual_default"
)
ks = evaluator.compute_ks()
pd.DataFrame(
    {"score": list(ks), "KS": [round(v, 3) for v in ks.values()]}
).sort_values("KS", ascending=False).reset_index(drop=True)

,score,KS
0,score_5,0.300
1,score_3,0.277
2,score_2,0.276
3,score_4,0.276
4,legacy_score,0.260


`score_5` separates best, and it beats the incumbent's own `legacy_score` — the necessary
condition for a swap to be worth running. But KS is a ranking statistic; it does not say *where*
the challenger buys you amplitude. This decision matrix does: it groups the contracted book by
`score_5` risk grade and reads the default-rate spread top to bottom. A challenger earns its place
by widening that spread — pushing the best grade lower and the worst grade higher.

In [6]:
contracted = base[base["hired"] == 1].copy()
grade_matrix = fit_risk_groups(
    contracted, "score_5", "actual_default", bins=20, max_groups=5, min_vol_ratio=0.05
)
matrix = grade_matrix.groups.rename(columns={"risk_rating": "grade", "pd": "default_rate"})
matrix["default_rate"] = matrix["default_rate"].round(3)
print(matrix.to_string(index=False))
amplitude = matrix["default_rate"].max() - matrix["default_rate"].min()
print(f"\namplitude (worst - best grade): {amplitude:.3f}")

 grade  volume  default_rate
     1    1443         0.016
     2     570         0.035
     3    2286         0.074
     4     574         0.120
     5     846         0.176

amplitude (worst - best grade): 0.160


## 3. The swap — and the warning

Now approve on `score_5` instead of `legacy_score`. A deliberately low cutoff (500) admits a large
population the incumbent rejected: those are the **swap-ins**, and none of them ever contracted, so
none of them has an observed `actual_default`. The simulator has to impute a PD for every one.

Its default method is a *local score calibration*: bin the keep-ins (whose outcome you **do**
observe) by score, and give each swap-in the observed default rate of its bin. Reasonable — until
you notice the swap-ins live in a score range the keep-ins barely occupy. Watch what the library
does about it.

In [7]:
def challenger(score_5_cutoff=500, stress=None, **policy_kwargs):
    policy = (
        CreditPolicy(
            applicant_id_col="applicant_id",
            score_cols=("score_5",),
            current_approval_col="approved",
            actual_default_col="actual_default",
            **policy_kwargs,
        )
        .filter("Bureau knock-outs", (col("age") >= 18) & (col("vl_negativacao") <= 5000))
        .cutoff("Challenger cutoff", {"score_5": score_5_cutoff}, direction="gte")
        .rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")
    )
    # A declared conservatism markup on the swap-in PD (see section 5c). No stress by default.
    return policy.stress(stress) if stress else policy


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sim_naive = challenger().simulate(base, method="analytical")

print(sim_naive.data["scenario"].value_counts().to_string())
print()
for w in caught:
    print(f"[{w.category.__name__}] {w.message}\n")

scenario
keep_out    32178
swap_in     15494
keep_in     12294
swap_out       34

[CalibrationReliabilityWarning] Swap-in PD calibration is unreliable: 2 adjacent score bin(s) invert — PD moves against the score's declared direction, so lowering a gate can read as reducing default rate. Reduce CreditPolicy(calibration_bins=...), or calibrate by rating via calibration_score_col.

[UserWarning] Swap-in PD imputation is using the default of 10 score bins (deciles). Pass CreditPolicy(calibration_bins=...) to set a different granularity.



The `CalibrationReliabilityWarning` is not decoration. It fires because adjacent score bins
**invert** — PD moves *against* the score's declared direction — which means the imputation is not
measuring what it claims. Left alone, lowering a gate could read as *reducing* default. The library
refuses to let that pass silently.

## 4. Measuring the damage

Is the warning crying wolf? This is exactly what the `true_pd` oracle is for. Compare the imputed
swap-in PD against the truth no lender ever has.

In [8]:
def swap_in_pd(sim):
    d = sim.data
    si = d[d["scenario"] == "swap_in"]
    w = si["new_approval"]
    imputed = (si["simulated_default"] * w).sum() / w.sum()
    true = (si["true_pd"] * w).sum() / w.sum()
    return imputed, true


imputed, true = swap_in_pd(sim_naive)
print(f"swap-in PD, imputed: {imputed:.3f}")
print(f"swap-in PD, true:    {true:.3f}   (oracle)")
print(f"understated by:      {1 - imputed / true:.0%}")

swap-in PD, imputed: 0.154
swap-in PD, true:    0.218   (oracle)
understated by:      29%


The warning was right. The naive imputation understates swap-in risk by roughly a third — the same
failure mode ADR 0010 documents. You would approve this population believing it defaults far less
than it does. Now the remediations.

## 5. The two remediations — and the one that is not a fix

The warning names two levers, one per trigger. Take them in order.

### 5a. The inversion knob: `calibration_bins`

Adjacent inversion is a *granularity* symptom — too many bins over too few observations, so noise
outranks signal. Coarsen the grid and the inversion disappears. The warning clears.

In [9]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sim_bins = challenger(calibration_bins=5).simulate(base, method="analytical")

reliability = [w for w in caught if issubclass(w.category, CalibrationReliabilityWarning)]
print(f"reliability warnings now: {len(reliability)}")

imputed_b, true_b = swap_in_pd(sim_bins)
print(f"swap-in PD, imputed: {imputed_b:.3f}   (was {imputed:.3f})")
print(f"swap-in PD, true:    {true_b:.3f}")
print(f"understated by:      {1 - imputed_b / true_b:.0%}")

reliability warnings now: 0
swap-in PD, imputed: 0.141   (was 0.154)
swap-in PD, true:    0.216
understated by:      35%


Read that carefully. The **warning is gone**, but the **number barely moved** — still understated
by about a third. Silencing the alarm did not put out the fire. Coarser bins fixed the *inversion*;
they did nothing about the deeper problem, because the deeper problem is not granularity.

### 5b. What the bins were hiding: no overlap

The swap-ins were rejected *on the incumbent's axis*, `legacy_score`. Look at the calibration
through that axis and the real problem is undeniable. Point `calibration_score_col` at
`legacy_score` and the library raises a different member of the same warning family.

In [10]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    challenger(calibration_score_col="legacy_score").simulate(base, method="analytical")

for w in caught:
    if issubclass(w.category, CalibrationReliabilityWarning):
        print(f"[{w.category.__name__}] {w.message}\n")

ki = sim_naive.data.query("scenario == 'keep_in'")["legacy_score"]
si = sim_naive.data.query("scenario == 'swap_in'")["legacy_score"]
print(f"keep-in legacy_score range: [{ki.min():.0f}, {ki.max():.0f}]")
print(f"swap-in legacy_score range: [{si.min():.0f}, {si.max():.0f}]")

[CalibrationReliabilityWarning] Swap-in PD calibration is unreliable: 100% of swap-ins score outside the keep-ins' observed range, so their imputed PD is edge-clamp extrapolation, not measurement. No bin count fixes this — the score range has no observed defaults to measure. Pass an estimated_default_col with a model PD.



keep-in legacy_score range: [790, 999]
swap-in legacy_score range: [133, 789]


Every swap-in scores *below the entire keep-in range*. There are **no observed defaults in their
band to calibrate against** — the imputation is edge-clamp extrapolation, not measurement. As the
warning says, *no bin count fixes this.* `calibration_bins` was never going to work, because the
information simply is not in the contracted book.

### 5c. The answer: aggravate, don't infer

The warning's own text offers an escape — *pass an `estimated_default_col` with a model PD*. That
means importing an outside opinion about a population you never lent to: a bureau flag, another
lender's model. This masterclass declines it. External inference drags a different book's data into
your decision and re-opens exactly the circularity v0.5 spent itself closing; the honest posture
toward a population you *cannot measure* is not to borrow someone else's guess but to price your own
ignorance.

The package's self-contained answer is **aggravation**. Two facts are settled: you cannot measure
the reject, and the score-based baseline errs *low*. So do not ship it raw. `AggravationStress`
multiplies the reference-score baseline PD (`score_5`, the score already driving the calibration) by
a declared factor — a **policy decision about conservatism, not a fit to anything**. Clear the
inversion first (`calibration_bins=5`, from 5a) so the baseline is stable, then turn the dial and
watch where the swap-in PD lands against the oracle you never get in production.

In [11]:
dial = []
for factor in [1.0, 1.25, 1.5, 1.75, 2.0]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sim_f = challenger(
            calibration_bins=5, stress=(None if factor == 1.0 else factor)
        ).simulate(base, method="analytical")
    imputed_f, true_f = swap_in_pd(sim_f)
    dial.append({"stress_factor": factor, "swap_in_pd": round(imputed_f, 3)})

dial = pd.DataFrame(dial)
dial["true_pd (oracle)"] = round(true_f, 3)
dial

,stress_factor,swap_in_pd,true_pd (oracle)
0,1.00,0.141,0.216
1,1.25,0.176,0.216
2,1.50,0.212,0.216
3,1.75,0.247,0.216
4,2.00,0.282,0.216


Read the dial. At factor 1.0 (no markup, bins fixed) the PD is the same understated ~0.14 from 5a.
A flat **+50% markup** (factor 1.5) brings it to *parity* with the truth; every notch above buys
margin over it. And the whole column is **warning-free** — coarsening the bins in 5a already cleared
the inversion, and aggravation was never trying to silence anything.

That is the honest trade. You are not claiming to know the reject's PD — you cannot, and nothing
here pretends to. You are refusing to underwrite an unmeasurable population at an optimistic number,
and the factor is the lever you turn toward caution. `+50%` is a defensible starting posture; a
risk-averse desk would set it higher. From here the challenger carries `calibration_bins=5` and a
`stress(1.5)` markup — no external data, all from the reference score.

## 6. Setting the cutoff honestly

500 was a strawman to force swap-ins. The real cutoff comes from the sweep engine. `optimize_cutoffs`
sweeps `score_5`, holding every other stage fixed, and returns the frontier. Two v0.5 rules apply:

- **Direction is declared, never inferred** — `directions={"score_5": "gte"}` (keep scores *at or
  above* the cutoff). There is no guessing which way a score points.
- **The config always binds.** The hard filters, the take-up stage, the `calibration_bins=5`
  baseline and the `stress(1.5)` markup from section 5 all hold at every grid point; only the swept
  cutoff moves.

In [12]:
config = (
    CreditPolicy(
        applicant_id_col="applicant_id",
        score_cols=("score_5",),
        current_approval_col="approved",
        actual_default_col="actual_default",
        calibration_bins=5,
    )
    .filter("Bureau knock-outs", (col("age") >= 18) & (col("vl_negativacao") <= 5000))
    .rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")
    .stress(1.5)
)

with warnings.catch_warnings():
    # The sweep widens the swap-in baseline (it removes the cutoff), so the
    # section-5 reliability caveat re-fires. The config already answers it —
    # calibration_bins=5 + stress(1.5) — so we mute it here rather than re-teach it.
    warnings.simplefilter("ignore", CalibrationReliabilityWarning)
    opt = optimize_cutoffs(
        base,
        config,
        cutoff_steps=15,
        target_default_rate=0.09,
        min_approval_rate=0.20,
        directions={"score_5": "gte"},
    )
print("best cutoff:", {k: round(v) for k, v in opt.best_combination.items()})
print("at metrics:", {k: round(v, 3) for k, v in opt.metrics.items() if isinstance(v, float)})
opt.pareto_frontier[["score_5", "overall_approval_rate", "overall_default_rate"]].round(3)

best cutoff: {'score_5': 751}
at metrics: {'overall_approval_rate': 0.242, 'overall_default_rate': 0.088, 'tradeoff_score': -0.197}


,score_5,overall_approval_rate,overall_default_rate
0,941.000,0.048,0.014
1,877.786,0.116,0.037
2,814.571,0.181,0.059
3,751.357,0.242,0.088
4,688.143,0.300,0.116
5,624.929,0.357,0.135
6,561.714,0.410,0.148
7,498.500,0.464,0.159
8,435.286,0.516,0.167
9,372.071,0.570,0.173


`find_equivalent` answers the manager's question — *"give me the cutoff that lands near 30%
approval"* — off the same grid, no re-run. It returns a *different* operating point from the
constrained best above, without re-sweeping.

In [13]:
opt.find_equivalent("approval_rate", target_value=0.30, tolerance=0.02)[
    ["score_5", "overall_approval_rate", "overall_default_rate"]
].round(3)

,score_5,overall_approval_rate,overall_default_rate
0,688.143,0.3,0.116


One national cutoff is rarely right, because risk is not uniform across regions. The sweep engine is
cheap enough to run once per region — the same declared-direction call, on each slice.

In [14]:
region_rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for region, sub in base.groupby("region"):
        o = optimize_cutoffs(
            sub, config, cutoff_steps=15, target_default_rate=0.12,
            min_approval_rate=0.15, directions={"score_5": "gte"},
        )
        region_rows.append({
            "region": region,
            "applicants": len(sub),
            "cutoff": round(o.best_combination["score_5"]),
            "approval_rate": round(o.metrics["overall_approval_rate"], 3),
            "default_rate": round(o.metrics["overall_default_rate"], 3),
        })
pd.DataFrame(region_rows).sort_values("cutoff").reset_index(drop=True)

,region,applicants,cutoff,approval_rate,default_rate
0,Sul,12104,567,0.422,0.117
1,Norte,4125,683,0.273,0.116
2,Sudeste,26989,691,0.304,0.112
3,Nordeste,10812,743,0.228,0.113
4,Centro-Oeste,5970,747,0.239,0.104


Nordeste has to cut hardest for the same 12% default target — exactly the regional risk bias baked
into the book. A single national threshold would over-lend there and under-lend elsewhere.

## 7. Sequential cutoffs — buying data only when it pays

Not every score is free. Suppose `score_5` is in-house but `score_4` is a paid bureau pull. Buying
`score_4` for *every* applicant is waste — most are decided by `score_5` alone. The move is
**sequential**: gate on the free score first, and buy the expensive one only for the survivors — the
"tall grass" you still cannot see into. First, how many survive the free gate.

In [15]:
free_gate = round(opt.best_combination["score_5"])
survivors = base[base["score_5"] >= free_gate]
print(f"free gate score_5 >= {free_gate}")
print(f"survivors needing a score_4 pull: {len(survivors):,} of {len(base):,} "
      f"({len(survivors) / len(base):.1%})")
print(f"API calls avoided:                {1 - len(survivors) / len(base):.1%}")

free gate score_5 >= 751
survivors needing a score_4 pull: 15,599 of 60,000 (26.0%)
API calls avoided:                74.0%


Does the second pull actually separate risk *among the survivors*, or is it redundant once
`score_5` has spoken? A **bivariate** grouping answers it — cluster the survivors on both scores at
once and read whether the grades still spread.

In [16]:
survivors_contracted = survivors[survivors["hired"] == 1].copy()
bivariate = fit_risk_groups(
    survivors_contracted, ["score_5", "score_4"], "actual_default",
    bins=12, max_groups=5, min_vol_ratio=0.05,
)
biv = bivariate.groups.rename(columns={"risk_rating": "grade", "pd": "default_rate"})
biv["default_rate"] = biv["default_rate"].round(3)
print(biv.to_string(index=False))
spread = biv["default_rate"].max() - biv["default_rate"].min()
print(f"\nspread among score_5 survivors: {spread:.3f}")

 grade  volume  default_rate
     1    1204         0.007
     2    1489         0.034
     3     671         0.067
     4     905         0.091
     5     567         0.162

spread among score_5 survivors: 0.155


The grades still spread after `score_5` has filtered — so the second pull earns its cost on the
survivors, and *only* there. That is the whole case for a sequential funnel over a flat one.

## 8. Rating, validated out of time

A cutoff is a yes/no; a **rating** is the price. `fit_risk_groups` clusters the contracted book into
risk grades, but a grade is only useful if it *holds up on data it was not fit on*. Pass `oot_date`
and the fit splits the book at that vintage — grades are learned on the training vintages and the
report shows their default rate on the held-out ones.

In [17]:
rating = fit_risk_groups(
    contracted, "score_5", "actual_default",
    bins=20, max_groups=5, min_vol_ratio=0.05, max_crossings=3,
    time_col="safra", oot_date="2025-01",
)
letters = {1: "A", 2: "B", 3: "C", 4: "D", 5: "E"}
report = rating.report.copy()
report["grade"] = report["risk_rating"].map(letters)
report["pd"] = report["pd"].round(3)
print(f"grades: {rating.n_groups}")
report.pivot(index="grade", columns="period", values="pd")[["Train", "OOT"]]

grades: 5


period,Train,OOT
grade,,
A,0.014,0.024
B,0.052,0.081
C,0.078,0.078
D,0.124,0.112
E,0.176,0.177


The default ladder is monotone on Train, and the OOT column stays in the same band grade by grade —
the grades hold up out of time rather than being an artefact of the fit. (Where a grade *did* drift,
this report is exactly where you would catch it.) That validation is the licence to *use* the rating
inside a policy. Two ways to use it:

- As a **price** — attach it with `.with_rating(recipe)` so every decision carries a grade.
- As a **gate** — exclude the worst grade outright, a rating-as-filter stage.

In [18]:
recipe = rating.recipe
base["Rating"] = recipe.predict(base)["risk_rating"].map(letters)

rated_policy = (
    CreditPolicy(
        applicant_id_col="applicant_id",
        score_cols=("score_5",),
        current_approval_col="approved",
        actual_default_col="actual_default",
        calibration_bins=5,
    )
    .filter("Bureau knock-outs", (col("age") >= 18) & (col("vl_negativacao") <= 5000))
    .cutoff("Challenger cutoff", {"score_5": free_gate}, direction="gte")
    .filter("Rating exclusion", col("Rating") != "E")
    .rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")
    .with_rating(recipe)
    .stress(1.5)
)
sim_rated = rated_policy.simulate(base, method="analytical")
for name, value in kpis(sim_rated).items():
    print(f"{name:>14}: {value:.3f}")

 approval_rate: 0.242
  take_up_rate: 0.463
  default_rate: 0.082


## 9. A rate stage that is *not* take-up

Take-up is a `RateStage`. So is an anti-fraud screen. In v0.5 they are the **same object** — a rate
stage is a rate stage; the name is a label, not a behaviour. `passed_antifraud` is a Bernoulli(0.9)
outcome in the book, independent of risk: a fraud screen expressed as data. It plugs in exactly like
take-up did, `observed_col` and all — the only difference is `calibrate_by=None`, because a fraud
pass rate is not calibrated by risk score.

In [19]:
policy_full = (
    CreditPolicy(
        applicant_id_col="applicant_id",
        score_cols=("score_5",),
        current_approval_col="approved",
        actual_default_col="actual_default",
        calibration_bins=5,
    )
    .filter("Bureau knock-outs", (col("age") >= 18) & (col("vl_negativacao") <= 5000))
    .cutoff("Challenger cutoff", {"score_5": free_gate}, direction="gte")
    .filter("Rating exclusion", col("Rating") != "E")
    .rate("Anti-fraud", base_rate=1.0, observed_col="passed_antifraud", calibrate_by=None)
    .rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")
    .with_rating(recipe)
    .stress(1.5)
)
sim_full = policy_full.simulate(base, method="analytical")
for name, value in kpis(sim_full).items():
    print(f"{name:>14}: {value:.3f}")

 approval_rate: 0.242
  take_up_rate: 0.415
  default_rate: 0.083


Two rate stages, one interface. `take_up_rate` now folds *both* — the anti-fraud screen and take-up
compose. Nothing in the engine distinguishes "take-up" from "fraud" except the string you named
them, which is the point: v0.5 stopped treating stage names as semantics.

## 10. Suggesting the hard filters

So far the bureau knock-outs were hand-picked. `suggest_hard_filters` proposes them instead — a
**consultative** tool that scores every declared candidate against the approval budget and returns a
greedy set for a human to parametrise, never an automated decision.

The subtlety is measurement, and it is the same mask as before. A real hard filter targets people
the incumbent *already rejected*, whose `actual_default` is `NaN`. Staying true to the no-inference
rule of section 5, we do **not** borrow an outside flag to see them — we measure lift where the
outcome is genuinely observed, the **contracted** book, and let the tool warn that this population
already survived the incumbent's screen. Observed lift on survivors *understates* a real filter, so
a candidate that clears the bar here is, if anything, stronger than it looks.

In [20]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # mutes the coverage caveat; we discuss it above
    hf = suggest_hard_filters(
        base,
        bad_col="actual_default",
        directions={
            "vl_negativacao": "lte",
            "vl_vencido_scr": "lte",
            "vl_protestos": "lte",
            "age": "gte",
        },
        hf_approval_floor=0.50,
        lift_min=1.3,
    )

print("budget:", {k: round(v, 3) for k, v in hf.budget.items()})
print("\nsuggested set (greedy, in selection order):")
print(hf.rule_set[
    ["column", "direction", "threshold", "iv", "lift", "bads_rejected", "cumulative_cut"]
].round(3).to_string(index=False))
print("\nrejected candidates, with the reason each lost:")
print(hf.rejected[["column", "iv", "lift", "reason"]].round(3).to_string(index=False))

budget: {'floor': 0.5, 'spent': 0.463, 'approval_rate': 0.537, 'headroom': 0.037}

suggested set (greedy, in selection order):
        column direction  threshold    iv  lift  bads_rejected  cumulative_cut
vl_negativacao       lte        0.0 0.035 1.460           80.0           0.217
vl_vencido_scr       lte        0.0 0.087 1.704           79.0           0.351
           age       gte       25.0 0.026 1.365           47.0           0.463

rejected candidates, with the reason each lost:
      column    iv  lift                                          reason
vl_protestos 0.052 2.203 marginal cut 6.0% exceeds remaining budget 3.7%


Every rejected candidate comes back with the reason it lost — too little lift, or a cut the approval
budget could not afford. That transparency is the tool's whole purpose: it hands you an auditable
starting set to parametrise, not a black-box verdict.

## 11. The P&L, and export

The executive question is one table: against the incumbent, what does the challenger do to approval,
to bad rate, to volume? `print_delta_table` reads it off the two simulations directly. Swap-in bad
rates here rest on the **aggravated reference-score baseline** from section 5 (`calibration_bins=5`
plus `stress(1.5)`) — the conservative in-package estimate, not the naive imputation the warning
flagged.

In [21]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sim_challenger = challenger(
        score_5_cutoff=free_gate, calibration_bins=5, stress=1.5
    ).simulate(base, method="analytical")

print_delta_table(sim_challenger, sim_incumbent)

=== DELTA TABLE: EXECUTIVE P&L ===
Metric                                  Legacy         New   Delta Abs   Delta Rel
------------------------------------------------------------------------------
Global Approval Rate (% ToF)            20.55%      24.25%      +3.70%      +18.0%
Expected Bad Rate                        7.54%       8.01%      +0.47%       +6.3%
Expected Hired Volume                    5,719       6,740      +1,021      +17.9%


And the swap breakdown behind that P&L — who moves in, who moves out, and the default rate each
quadrant carries (`keep_out` stays `NaN`: rejected by both policies, so no outcome exists to report).

In [22]:
swaps = summarize_results(sim_challenger)
for c in ["Applicants", "Approved", "Hired"]:
    swaps[c] = swaps[c].round().astype(int)
swaps["Bad_Rate"] = swaps["Bad_Rate"].round(3)
swaps

,scenario,Applicants,Approved,Hired,Bad_Rate
0,keep_in,10817,10817,4836,0.058
1,keep_out,43940,0,0,NaN
2,swap_in,3732,3732,1904,0.137
3,swap_out,1511,0,0,0.172


Once the challenger is chosen, `export` freezes it — the stages, the cutoff, the rating recipe — into
a `DeploymentPolicy` you can serialise and ship. `clean=True` drops study-only metadata and keeps the
hard rules and cutoffs a production system actually enforces.

In [23]:
deployment = policy_full.export(clean=True)
print(type(deployment).__name__, "ready to serialise")

DeploymentPolicy ready to serialise


## Where this leaves you

The long road ends where the short one pointed. You met the swap, heard the calibration warning fire
for real, measured that it was telling the truth, and answered it the honest way — not by importing
an outside opinion, but by pricing your own ignorance: a conservative markup on the reference-score
baseline for a population you cannot measure. Everything after that — the declared-direction sweep,
the sequential funnel, the out-of-time rating, the composed rate stages, the suggested filters, the
P&L — is ordinary work once the one hard thing is faced.

The masterclass's whole claim is that the hard thing is *reachable*: the library makes the dishonest
path loud and the honest path short.